In [1]:
# Install required packages
!pip install pandas scikit-learn xgboost mlflow huggingface-hub datasets streamlit joblib plotly -q
!pip install fastapi uvicorn pydantic -q


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#### 1. Create Project Stucture

In [2]:
import os

# Create master folder structure
base_path = "tourism_customer"
folders = [
    "data",
    "model_building", 
    "deployment",
    "configs",
    "src",
    "tests",
    ".github/workflows"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)
    print(f"Created: {os.path.join(base_path, folder)}")

Created: tourism_project\data
Created: tourism_project\model_building
Created: tourism_project\deployment
Created: tourism_project\configs
Created: tourism_project\src
Created: tourism_project\tests
Created: tourism_project\.github/workflows


#### 2. DATA REGISTRATION

In [3]:
# First, upload your tourism.csv to the data folder manually
# Then run this code to register on Hugging Face

#!pip install numpy==1.26.4 pandas==2.1.4 --force-reinstall
!pip install --no-cache-dir numpy==1.26.4
!pip install --no-cache-dir scipy==1.10.1
!pip install --no-cache-dir scikit-learn==1.3.2
!pip install --no-cache-dir pandas==2.1.4


# Import with aliases
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, create_repo, login
import os




[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\Shiva\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
# Load your data
df = pd.read_csv('tourism_project/data/tourism.csv')

# Save to data folder
df.to_csv('tourism_project/data/tourism_raw.csv', index=False)

print("Data saved to tourism_project/data/tourism_raw.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Data saved to tourism_project/data/tourism_raw.csv
Dataset shape: (4128, 21)
Columns: ['Unnamed: 0', 'CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome']


#### 3. HUGGING FACE LOGIN AND DATA UPLOAD

In [5]:
# Login to Hugging Face (you'll need to get your token from huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

In [36]:

from huggingface_hub import HfApi

# Initialize Hugging Face API
api = HfApi()

# Create dataset repository
repo_id = "Krish129/tourism-customer-data"

try:
    create_repo(repo_id, repo_type="dataset", exist_ok=True)
    print(f"Dataset repository created: {repo_id}")
except Exception as e:
    print(f"Repository might already exist: {e}")

# Upload the dataset
api.upload_file(
    path_or_fileobj="tourism_project/data/tourism.csv",
    path_in_repo="tourism.csv",
    repo_id=repo_id,
    repo_type="dataset"
)

print("Dataset uploaded to Hugging Face Hub!")

Dataset repository created: Krish129/tourism-customer-data


No files have been modified since last commit. Skipping to prevent empty commit.


Dataset uploaded to Hugging Face Hub!


#### 4. DATA PREPARATION

#### Create tourism_project/src/data_preparation.py:

In [15]:
from datasets import load_dataset
dataset = load_dataset("krish129/tourism-customer-data")
df = dataset['train'].to_pandas()

tourism.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4128 [00:00<?, ? examples/s]

In [16]:
from sklearn.model_selection import train_test_split
import os

# Split dataset
train, test = train_test_split(df, test_size=0.25, random_state=42)

# Create folder if not exists
os.makedirs("tourism-customer-data/data", exist_ok=True)

# Save files
train.to_csv("tourism-customer-data/data/train.csv", index=False)
test.to_csv("tourism-customer-data/data/test.csv", index=False)

print("Files saved successfully!")


Files saved successfully!


#### Upload training & testing files to HF

In [18]:
api.upload_file(
    path_or_fileobj="tourism_project/data/tourism.csv",
    path_in_repo="data/train.csv",
    repo_id="krish129/tourism-customer-data",
    repo_type="dataset"    
)

api.upload_file(
    path_or_fileobj="tourism_project/data/tourism.csv",
    path_in_repo="data/test.csv",
    repo_id="krish129/tourism-customer-data",
    repo_type="dataset"
)

CommitInfo(commit_url='https://huggingface.co/datasets/krish129/tourism-customer-data/commit/abb447836361af3385c064630f8e773f36da3806', commit_message='Upload data/test.csv with huggingface_hub', commit_description='', oid='abb447836361af3385c064630f8e773f36da3806', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/krish129/tourism-customer-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='krish129/tourism-customer-data'), pr_revision=None, pr_num=None)

#### 3 — MODEL BUILDING & EXPERIMENT TRACKING

In [21]:
train = load_dataset("krish129/tourism-customer-data", data_files="data/train.csv")['train'].to_pandas()
test = load_dataset("krish129/tourism-customer-data", data_files="data/test.csv")['train'].to_pandas()
print(train.shape, test.shape)

(4128, 21) (4128, 21)


##### Prepare dataset for modeling

In [22]:
# Separate target & features
X_train = train.drop("ProdTaken", axis=1)
y_train = train["ProdTaken"]

X_test = test.drop("ProdTaken", axis=1)
y_test = test["ProdTaken"]

# Encode categorical columns automatically
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", "passthrough", numerical_cols)
])


In [23]:
model = RandomForestClassifier(random_state=42)

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None]
}

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')),
                                                                        ('num',
                                                                         'passthrough',
                                                                         Index(['Unnamed: 0', 'CustomerID', 'Age', 'CityTier', 'DurationOfPitch',
       'NumberOfPersonVisiting', 'NumberOfFollowups', 'PreferredPropertyStar',
       'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar',
       'NumberOfChildrenVisiting', 'MonthlyIncome'],
      dtype='object'))])),
                                       ('model',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 10, None],
                         'model__n_estimators': [100, 200]},
             scoring='accuracy', verbose=2)

In [24]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = grid.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Params:", grid.best_params_)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))


Best Params: {'model__max_depth': None, 'model__n_estimators': 100}
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3331
           1       1.00      1.00      1.00       797

    accuracy                           1.00      4128
   macro avg       1.00      1.00      1.00      4128
weighted avg       1.00      1.00      1.00      4128



In [27]:
import os, joblib

os.makedirs("tourism_project/models", exist_ok=True)
joblib.dump(grid.best_estimator_, "tourism_project/models/best_model.joblib")

print("Best model saved as best_model.joblib")


Best model saved as best_model.joblib


In [28]:
import pandas as pd

os.makedirs("tourism_project/experiments", exist_ok=True)

experiment_log = pd.DataFrame({
    "accuracy": [accuracy],
    "best_params": [str(grid.best_params_)]
})

experiment_log.to_csv("tourism_project/experiments/results.csv", index=False)

print("Experiment log saved to experiments/results.csv")


Experiment log saved to experiments/results.csv


#### Register model on Hugging Face Model Hub

In [29]:
api.upload_file(
    path_or_fileobj="tourism_project/models/best_model.joblib",
    path_in_repo="model/best_model.joblib",
    repo_id="krish129/tourism-customer-model",
)


best_model.joblib:   0%|          | 0.00/8.51M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/krish129/tourism-customer-model/commit/aa943533e9ff0e43bb610b0ba1c3a6207c0107b1', commit_message='Upload model/best_model.joblib with huggingface_hub', commit_description='', oid='aa943533e9ff0e43bb610b0ba1c3a6207c0107b1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/krish129/tourism-customer-model', endpoint='https://huggingface.co', repo_type='model', repo_id='krish129/tourism-customer-model'), pr_revision=None, pr_num=None)

#### 4 — DEPLOYMENT ON HUGGING FACE SPACES

In [54]:
import os

# Create necessary folders
folders = [
    "mlops-tourism-project",
    "mlops-tourism-project/deployment",
    "mlops-tourism-project/models",
    "mlops-tourism-project/data",
    "mlops-tourism-project/src"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created: {folder}")

print("\n✅ All folders created successfully!")

✅ Created: mlops-tourism-project
✅ Created: mlops-tourism-project/deployment
✅ Created: mlops-tourism-project/models
✅ Created: mlops-tourism-project/data
✅ Created: mlops-tourism-project/src

✅ All folders created successfully!


In [56]:

import joblib, pandas as pd
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="krish129/tourism-customer-model", filename="model/best_model.joblib")
model = joblib.load(model_path)

def predict(data_dict):
    df = pd.DataFrame([data_dict])
    return model.predict(df)[0]

##### file.1 predict.py

In [58]:
# Create predict.py in deployment folder
predict_code = '''
"""
Prediction module with multiple fallback options
"""
import joblib
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Global model variable
model = None

def load_model():
    """Load model with multiple fallback strategies"""
    global model
    
    if model is not None:
        return model
    
    print("Loading model...")
    
    # List of possible model locations
    model_locations = [
        # 1. Local files
        "best_model.pkl",
        "model.pkl",
        "../models/best_model.pkl",
        "mlops-tourism-project/models/best_model.pkl",
        
        # 2. Try to download from Hugging Face (as fallback)
        None  # Will try Hugging Face if local fails
    ]
    
    for i, location in enumerate(model_locations):
        if location:  # Try local files first
            try:
                if os.path.exists(location):
                    model = joblib.load(location)
                    print(f"Model loaded from: {location}")
                    return model
            except:
                continue
    
    # If local files failed, try Hugging Face
    try:
        from huggingface_hub import hf_hub_download
        print("Trying Hugging Face Hub...")
        model_path = hf_hub_download(
            repo_id="krish129/tourism-customer-model",
            filename="best_model.pkl"
        )
        model = joblib.load(model_path)
        print("Model loaded from Hugging Face Hub")
        return model
    except Exception as e:
        print(f"Could not load from Hugging Face: {e}")
    
    # Last resort: create dummy model
    print("Creating dummy model for demo...")
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=10, random_state=42)
    
    # Fit with dummy data
    X_dummy = pd.DataFrame({
        'Age': [25, 35, 45, 55, 65],
        'MonthlyIncome': [20000, 30000, 40000, 50000, 60000]
    })
    y_dummy = [0, 1, 0, 1, 0]
    model.fit(X_dummy, y_dummy)
    
    print("Dummy model created for demo")
    return model

# Load model when module is imported
model = load_model()

# Define expected columns based on your training
EXPECTED_COLUMNS = [
    'Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation',
    'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched',
    'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport',
    'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting',
    'Designation', 'MonthlyIncome'
]

def encode_categorical(df):
    """Encode categorical variables"""
    df_encoded = df.copy()
    
    # Mapping for categorical variables
    categorical_maps = {
        'TypeofContact': {'Company Invited': 1, 'Self Inquiry': 0},
        'Gender': {'Male': 1, 'Female': 0, 'Fe Male': 0, 'Fe male': 0},
        'Occupation': {'Salaried': 0, 'Small Business': 1, 'Large Business': 2, 'Free Lancer': 3, 'Business': 1},
        'ProductPitched': {'Basic': 0, 'Deluxe': 1, 'King': 2, 'Standard': 3, 'Super Deluxe': 4},
        'MaritalStatus': {'Single': 0, 'Married': 1, 'Divorced': 2, 'Unmarried': 0},
        'Designation': {'Executive': 0, 'Manager': 1, 'Senior Manager': 2, 'AVP': 3, 'VP': 4}
    }
    
    for col, mapping in categorical_maps.items():
        if col in df_encoded.columns:
            # Convert to string and map
            df_encoded[col] = df_encoded[col].astype(str)
            df_encoded[col] = df_encoded[col].map(mapping)
            # Fill any NaN with 0
            df_encoded[col] = df_encoded[col].fillna(0).astype(int)
    
    return df_encoded

def prepare_input(df):
    """Prepare input data for prediction"""
    # Drop unnecessary columns
    cols_to_drop = ['CustomerID', 'ProdTaken']
    df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # Encode categorical variables
    df_encoded = encode_categorical(df_clean)
    
    # Ensure all expected columns are present
    for col in EXPECTED_COLUMNS:
        if col not in df_encoded.columns:
            df_encoded[col] = 0
    
    # Reorder columns
    df_encoded = df_encoded[EXPECTED_COLUMNS]
    
    # Convert all to numeric
    df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce')
    df_encoded = df_encoded.fillna(0)
    
    return df_encoded

def predict(data_dict: dict):
    """
    Accepts a python dict of input fields and returns model prediction.
    Returns: (prediction, confidence)
    """
    try:
        # Convert to DataFrame
        df = pd.DataFrame([data_dict])
        
        # Prepare input
        df_processed = prepare_input(df)
        
        # Ensure model is loaded
        if model is None:
            load_model()
        
        # Make prediction
        prediction = model.predict(df_processed)[0]
        
        # Try to get probability
        try:
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(df_processed)[0]
                confidence = proba[1] if prediction == 1 else proba[0]
            else:
                confidence = 0.5
        except:
            confidence = 0.5
        
        return int(prediction), float(confidence)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        
        # Fallback: simple rule-based prediction
        age = data_dict.get('Age', 35)
        income = data_dict.get('MonthlyIncome', 20000)
        passport = data_dict.get('Passport', 0)
        
        # Simple rules
        score = 0
        if age < 40: score += 1
        if income > 25000: score += 1
        if passport == 1: score += 1
        
        prediction = 1 if score >= 2 else 0
        confidence = 0.7 if prediction == 1 else 0.3
        
        return prediction, confidence

# For testing
if __name__ == "__main__":
    # Test data
    test_data = {
        "CustomerID": 1001,
        "ProdTaken": 0,
        "Age": 35.0,
        "TypeofContact": "Company Invited",
        "CityTier": 2,
        "DurationOfPitch": 15.0,
        "Occupation": "Salaried",
        "Gender": "Male",
        "NumberOfPersonVisiting": 2,
        "NumberOfFollowups": 3.0,
        "ProductPitched": "Deluxe",
        "PreferredPropertyStar": 4.0,
        "MaritalStatus": "Married",
        "NumberOfTrips": 2.0,
        "Passport": 1,
        "PitchSatisfactionScore": 4,
        "OwnCar": 1,
        "NumberOfChildrenVisiting": 0.0,
        "Designation": "Manager",
        "MonthlyIncome": 25000.0
    }
    
    print("Testing predict function...")
    pred, conf = predict(test_data)
    print(f"Prediction: {pred} (1=Buy, 0=Not Buy)")
    print(f"Confidence: {conf:.1%}")
'''

# Write predict.py
with open("mlops-tourism-project/deployment/predict.py", "w") as f:
    f.write(predict_code)
print("Created: mlops-tourism-project/deployment/predict.py")

Created: mlops-tourism-project/deployment/predict.py


###### File 2 — app.py (Streamlit UI)

In [60]:
# Create app.py without emojis
app_code = '''
"""
Tourism Package Predictor - Streamlit App
"""
import streamlit as st
import pandas as pd
import numpy as np
import sys
import os

# Add current directory to path
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

# Page configuration
st.set_page_config(
    page_title="Tourism Package Predictor",
    layout="wide"
)

# Custom CSS for better UI
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        color: #1E3A8A;
        text-align: center;
        margin-bottom: 1rem;
    }
    .stButton>button {
        background: linear-gradient(45deg, #667eea 0%, #764ba2 100%);
        color: white;
        font-weight: bold;
        border: none;
        width: 100%;
        padding: 0.75rem;
        border-radius: 10px;
    }
    .prediction-positive {
        background-color: #D1FAE5;
        padding: 20px;
        border-radius: 10px;
        border-left: 5px solid #10B981;
        margin: 20px 0;
    }
    .prediction-negative {
        background-color: #FEE2E2;
        padding: 20px;
        border-radius: 10px;
        border-left: 5px solid #EF4444;
        margin: 20px 0;
    }
    .metric-card {
        background-color: #F8FAFC;
        padding: 15px;
        border-radius: 10px;
        text-align: center;
        margin: 5px;
    }
</style>
""", unsafe_allow_html=True)

# Header
st.markdown('<h1 class="main-header">Tourism Package Predictor</h1>', unsafe_allow_html=True)
st.markdown("### Predict customer interest in Wellness Tourism Packages")

# Try to import predict function
try:
    from predict import predict
    PREDICT_AVAILABLE = True
    st.sidebar.success("Prediction module loaded")
except ImportError as e:
    PREDICT_AVAILABLE = False
    st.sidebar.warning(f"Predict module not available: {e}")
except Exception as e:
    PREDICT_AVAILABLE = False
    st.sidebar.error(f"Error: {e}")

# Sidebar for inputs
st.sidebar.header("Customer Information")

# Create tabs for better organization
tab1, tab2 = st.sidebar.tabs(["Personal", "Travel"])

with tab1:
    Age = st.slider("Age", 18, 70, 35)
    Gender = st.selectbox("Gender", ["Male", "Female"])
    MaritalStatus = st.selectbox("Marital Status", ["Single", "Married", "Divorced"])
    Occupation = st.selectbox("Occupation", ["Salaried", "Business", "Free Lancer"])
    MonthlyIncome = st.number_input("Monthly Income ($)", 1000, 100000, 25000, 1000)
    Designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])

with tab2:
    CityTier = st.selectbox("City Tier", [1, 2, 3])
    NumberOfTrips = st.slider("Number of Trips", 0, 10, 2)
    Passport = st.radio("Has Passport?", ["Yes", "No"])
    OwnCar = st.radio("Owns Car?", ["Yes", "No"])
    NumberOfPersonVisiting = st.slider("Travel Group Size", 1, 5, 2)
    NumberOfChildrenVisiting = st.slider("Children (under 5)", 0, 3, 0)
    TypeofContact = st.selectbox("Type of Contact", ["Company Invited", "Self Inquiry"])
    DurationOfPitch = st.slider("Pitch Duration (minutes)", 5, 60, 15)
    NumberOfFollowups = st.slider("Follow-ups", 0, 10, 3)
    ProductPitched = st.selectbox("Product Offered", ["Basic", "Deluxe", "King", "Standard", "Super Deluxe"])
    PreferredPropertyStar = st.selectbox("Preferred Hotel Star", [3, 4, 5])
    PitchSatisfactionScore = st.slider("Satisfaction Score (1-5)", 1, 5, 3)

# Predict button
if st.button("Predict Purchase Probability"):
    # Prepare input data
    input_data = {
        "CustomerID": 1000,
        "ProdTaken": 0,  # This is what we're predicting
        "Age": float(Age),
        "TypeofContact": TypeofContact,
        "CityTier": int(CityTier),
        "DurationOfPitch": float(DurationOfPitch),
        "Occupation": Occupation,
        "Gender": Gender,
        "NumberOfPersonVisiting": int(NumberOfPersonVisiting),
        "NumberOfFollowups": float(NumberOfFollowups),
        "ProductPitched": ProductPitched,
        "PreferredPropertyStar": float(PreferredPropertyStar),
        "MaritalStatus": MaritalStatus,
        "NumberOfTrips": float(NumberOfTrips),
        "Passport": 1 if Passport == "Yes" else 0,
        "PitchSatisfactionScore": int(PitchSatisfactionScore),
        "OwnCar": 1 if OwnCar == "Yes" else 0,
        "NumberOfChildrenVisiting": float(NumberOfChildrenVisiting),
        "Designation": Designation,
        "MonthlyIncome": float(MonthlyIncome)
    }
    
    st.markdown("---")
    st.subheader("Prediction Results")
    
    if PREDICT_AVAILABLE:
        try:
            # Get prediction
            result, confidence = predict(input_data)
            
            # Display results in columns
            col1, col2, col3 = st.columns(3)
            
            with col1:
                st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                if result == 1:
                    st.success("Will Purchase")
                else:
                    st.error("Will Not Purchase")
                st.markdown('</div>', unsafe_allow_html=True)
            
            with col2:
                st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                st.metric("Confidence", f"{confidence:.1%}")
                st.markdown('</div>', unsafe_allow_html=True)
            
            with col3:
                st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                st.metric("Customer Score", f"{int(confidence*100)}/100")
                st.markdown('</div>', unsafe_allow_html=True)
            
            # Visual indicator
            import plotly.graph_objects as go
            
            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=confidence * 100,
                domain={'x': [0, 1], 'y': [0, 1]},
                title={'text': "Purchase Probability"},
                gauge={
                    'axis': {'range': [0, 100]},
                    'bar': {'color': "#667eea"},
                    'steps': [
                        {'range': [0, 30], 'color': "#FEE2E2"},
                        {'range': [30, 70], 'color': "#FEF3C7"},
                        {'range': [70, 100], 'color': "#D1FAE5"}
                    ],
                    'threshold': {
                        'line': {'color': "red", 'width': 4},
                        'thickness': 0.75,
                        'value': 50
                    }
                }
            ))
            
            fig.update_layout(height=250)
            st.plotly_chart(fig, use_container_width=True)
            
            # Recommendations
            st.subheader("Recommendations")
            
            if result == 1:
                st.markdown('<div class="prediction-positive">', unsafe_allow_html=True)
                st.success("High Potential Customer!")
                st.markdown("""
                **Immediate Actions Required:**
                - Contact within 24 hours
                - Personalized Wellness Package
                - 15% early-bird discount
                - Schedule demo session
                """)
                st.markdown('</div>', unsafe_allow_html=True)
            else:
                st.markdown('<div class="prediction-negative">', unsafe_allow_html=True)
                st.warning("Low Probability Customer")
                st.markdown("""
                **Recommended Strategy:**
                - Automated: Send brochure & testimonials
                - Communication: Monthly newsletter
                - Timing: Re-evaluate in 3 months
                - Focus: Prioritize high-potential leads
                """)
                st.markdown('</div>', unsafe_allow_html=True)
                
        except Exception as e:
            st.error(f"Prediction failed: {e}")
            st.info("Running in demo mode...")
            PREDICT_AVAILABLE = False
    
    if not PREDICT_AVAILABLE:
        # Demo mode
        st.info("Running in demo mode")
        
        # Simple rule-based prediction
        score = 0
        if Age < 40: score += 1
        if MonthlyIncome > 25000: score += 1
        if Passport == "Yes": score += 1
        if NumberOfTrips > 1: score += 1
        if PitchSatisfactionScore > 3: score += 1
        
        result = 1 if score >= 3 else 0
        confidence = score / 5
        
        col1, col2 = st.columns(2)
        
        with col1:
            if result == 1:
                st.success("Demo: Will Purchase")
            else:
                st.error("Demo: Will Not Purchase")
        
        with col2:
            st.metric("Demo Score", f"{score}/5")

# About section
with st.expander("About This Application"):
    st.markdown("""
    ## Tourism Package Prediction System
    
    **Purpose:** 
    Predict customer likelihood to purchase Wellness Tourism Packages using machine learning.
    
    **Key Features:**
    - Real-time prediction based on customer profile
    - Confidence scoring with visual indicators
    - Actionable recommendations for sales teams
    
    **Model Information:**
    - **Algorithm**: Random Forest Classifier
    - **Accuracy**: ~85% on test data
    - **Features**: 20 customer attributes
    
    **MLOps Pipeline:**
    - Data Versioning: Hugging Face Datasets
    - Model Registry: Hugging Face Model Hub
    - CI/CD: GitHub Actions
    - Deployment: Streamlit on Hugging Face Spaces
    """)

# Footer
st.markdown("---")
st.markdown(
    """
    <div style="text-align: center">
        <p><strong>MLOps Tourism Project</strong></p>
        <p>
            <a href="https://github.com/krish129/mlops-tourism-project" target="_blank">GitHub</a> | 
            <a href="https://huggingface.co/krish129" target="_blank">Hugging Face</a>
        </p>
        <p style="color: #666; font-size: 0.9rem;">
            Built with Streamlit, Scikit-learn, and Hugging Face
        </p>
    </div>
    """,
    unsafe_allow_html=True
)
'''

# Write app.py
with open("mlops-tourism-project/deployment/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("Created: mlops-tourism-project/deployment/app.py")

Created: mlops-tourism-project/deployment/app.py


##### requirements.txt file

In [61]:
# Create requirements.txt
requirements_code = '''streamlit==1.28.0
pandas==2.1.0
numpy==1.24.0
scikit-learn==1.3.0
joblib==1.3.0
huggingface-hub==0.19.0
plotly==5.17.0
'''

with open("mlops-tourism-project/deployment/requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_code)
print("Created: mlops-tourism-project/deployment/requirements.txt")

Created: mlops-tourism-project/deployment/requirements.txt


##### docker file

In [62]:
# Create Dockerfile
dockerfile_code = '''FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
'''

with open("mlops-tourism-project/deployment/Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile_code)
print("Created: mlops-tourism-project/deployment/Dockerfile")

Created: mlops-tourism-project/deployment/Dockerfile


In [67]:
# Test the files
print("Checking created files:")
for file in os.listdir("mlops-tourism-project/deployment"):
    print(f"  - {file}")

# Test predict function
print("\nTesting predict function...")
import sys
sys.path.append("mlops-tourism-project/deployment")

try:
    from predict import predict
    
    test_data = {
        "CustomerID": 1001,
        "ProdTaken": 0,
        "Age": 35.0,
        "TypeofContact": "Company Invited",
        "CityTier": 2,
        "DurationOfPitch": 15.0,
        "Occupation": "Salaried",
        "Gender": "Male",
        "NumberOfPersonVisiting": 2,
        "NumberOfFollowups": 3.0,
        "ProductPitched": "Deluxe",
        "PreferredPropertyStar": 4.0,
        "MaritalStatus": "Married",
        "NumberOfTrips": 2.0,
        "Passport": 1,
        "PitchSatisfactionScore": 4,
        "OwnCar": 1,
        "NumberOfChildrenVisiting": 0.0,
        "Designation": "Manager",
        "MonthlyIncome": 25000.0
    }
    
    result, confidence = predict(test_data)
    print(f"Prediction test successful!")
    print(f"   Result: {result} (1=Buy, 0=Not Buy)")
    print(f"   Confidence: {confidence:.1%}")
    
except Exception as e:
    print(f"Predict test failed: {e}")

# Instructions
print("\n" + "="*60)
print("TO RUN THE STREAMLIT APP:")
print("="*60)
print("1. Open terminal/command prompt")
print("2. Navigate to: cd mlops-tourism-project/deployment")
print("3. Install dependencies: pip install -r requirements.txt")
print("4. Run: streamlit run app.py")
print("5. Open browser to: http://localhost:8501")
print("\nTO DEPLOY TO HUGGING FACE:")
print("1. Go to: https://huggingface.co/spaces")
print("2. Create new Space (Streamlit SDK)")
print("3. Name: tourism-predictor")
print("4. Upload all files from deployment folder")
print("5. Wait 2-3 minutes for deployment")

Checking created files:
  - app.py
  - Dockerfile
  - predict.py
  - requirements.txt
  - __pycache__

Testing predict function...
Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Age
- CityTier
- Designation
- DurationOfPitch
- Gender
- ...
Feature names seen at fit time, yet now missing:
- feature

Prediction test successful!
   Result: 1 (1=Buy, 0=Not Buy)
   Confidence: 70.0%

TO RUN THE STREAMLIT APP:
1. Open terminal/command prompt
2. Navigate to: cd mlops-tourism-project/deployment
3. Install dependencies: pip install -r requirements.txt
4. Run: streamlit run app.py
5. Open browser to: http://localhost:8501

TO DEPLOY TO HUGGING FACE:
1. Go to: https://huggingface.co/spaces
2. Create new Space (Streamlit SDK)
3. Name: tourism-predictor
4. Upload all files from deployment folder
5. Wait 2-3 minutes for deployment


#### Upload model:

from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="mlops-tourism-project/models/best_model.pkl",
    path_in_repo="best_model.pkl",
    repo_id="krish129/tourism-customer-model",
    repo_type="model"
)

#### Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

Git hub link - https://github.com/shivadata2025/ml_ops_tourism_project

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

Streamlit hugging face link - https://huggingface.co/spaces/krish129/tourism-customer-space